# 🚀 RAGForge: GRPO Reinforcement Learning Training (Google Colab / Kaggle)

This notebook trains a compact, high-reasoning language model (**`Qwen/Qwen2.5-1.5B-Instruct`**) using **Group Relative Policy Optimization (GRPO)** grounded on real-world technical RAG discourse.

### 🧠 How GRPO Works Here:
1. **Group Sampling ($G=4$):** For each technical prompt, the model generates 4 candidate reasoning paths.
2. **Multi-Objective Rule Rewards:**
   - **`reward_reasoning_format`**: Enforces structured `<think>...</think>` and `<answer>...</answer>` tags.
   - **`reward_rag_concept_density`**: Rewards domain-specific mechanisms (BM25, Reciprocal Rank Fusion, HNSW, Cross-Encoder, etc.).
   - **`reward_anti_repetition`**: Penalizes degenerate looping or empty completions.

### Step 1: Check GPU Acceleration
Ensure your Colab runtime is set to **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).

In [ ]:
!nvidia-smi

### Step 2: Install Dependencies

In [ ]:
!pip install --upgrade -q transformers trl peft accelerate polars pyarrow datasets

### Step 3: Upload `grpo_train_dataset.parquet`
Upload the prepared dataset from your local `data/processed/grpo_train_dataset.parquet`.

In [ ]:
import os
from google.colab import files

if not os.path.exists("grpo_train_dataset.parquet"):
    print("Please upload your data/processed/grpo_train_dataset.parquet:")
    uploaded = files.upload()
else:
    print("grpo_train_dataset.parquet already present!")

### Step 4: Define Multi-Objective Reward Functions

In [ ]:
import re
from typing import Any, Dict, List, Optional, Set

RAG_CONCEPT_TAXONOMY = {
    "retrieval": [
        "dense embedding", "bi-encoder", "cross-encoder", "reranker", "bm25",
        "sparse retrieval", "splade", "hybrid search", "reciprocal rank fusion", "rrf",
        "hnsw", "approximate nearest neighbor", "ann", "cosine similarity",
        "colbert", "hyde", "query expansion", "two-stage retrieval"
    ],
    "chunking_and_indexing": [
        "chunk size", "chunk overlap", "sliding window", "recursive character splitter",
        "semantic chunking", "parent document retriever", "sentence window retrieval",
        "hierarchical indexing", "metadata filtering", "vector database", "qdrant", "milvus", "chroma"
    ],
    "generation_and_grounding": [
        "in-context learning", "context stuffing", "lost in the middle", "faithfulness",
        "hallucination mitigation", "citation grounding", "system prompt", "attribution"
    ],
    "evaluation_and_benchmarks": [
        "ragas", "truelens", "deepeval", "context precision", "context recall",
        "faithfulness", "answer relevance", "llm-as-a-judge", "mrr", "ndcg", "hit rate"
    ],
    "agentic_and_graph_rag": [
        "agentic rag", "graph rag", "knowledge graph", "neo4j", "cypher",
        "adaptive routing", "corrective rag", "crag", "self-rag", "tool calling", "multi-hop reasoning"
    ],
    "tooling_and_frameworks": [
        "langchain", "llamaindex", "haystack", "dspy", "ollama", "vllm", "fastapi"
    ]
}
ALL_RAG_CONCEPTS = {t for sub in RAG_CONCEPT_TAXONOMY.values() for t in sub}

def extract_text(completion: Any) -> str:
    """Extract text string whether completion is a str, dict, or list of message dicts."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        parts = []
        for item in completion:
            if isinstance(item, dict):
                parts.append(str(item.get("content", "")))
            elif isinstance(item, str):
                parts.append(item)
            else:
                parts.append(str(item))
        return "\n".join(parts)
    if isinstance(completion, dict):
        return str(completion.get("content", ""))
    return str(completion)

def normalize_text(text: Any) -> str:
    text_str = extract_text(text).lower()
    text_str = re.sub(r"[^\w\s-]", " ", text_str)
    return re.sub(r"\s+", " ", text_str).strip()

def reward_reasoning_format(completions: List[Any], **kwargs) -> List[float]:
    rewards = []
    for item in completions:
        text = extract_text(item)
        has_open = "<think>" in text
        has_close = "</think>" in text
        if has_open and has_close:
            match = re.search(r"<think>(.*?)</think>", text, flags=re.DOTALL)
            if match and len(match.group(1).strip()) >= 40:
                rewards.append(1.0)
            else:
                rewards.append(0.5)
        elif has_open or has_close:
            rewards.append(-0.5)
        else:
            rewards.append(-1.0)
    return rewards

def reward_rag_concept_density(completions: List[Any], taxonomy_topic: Optional[List[str]] = None, **kwargs) -> List[float]:
    rewards = []
    for idx, item in enumerate(completions):
        text = extract_text(item)
        norm_text = normalize_text(text)
        score = 0.0
        matched = set()
        cat = taxonomy_topic[idx] if taxonomy_topic and idx < len(taxonomy_topic) else ""
        for term in RAG_CONCEPT_TAXONOMY.get(cat, []):
            if term in norm_text and term not in matched:
                matched.add(term)
                score += 0.4
        for term in ALL_RAG_CONCEPTS:
            if term in norm_text and term not in matched:
                matched.add(term)
                score += 0.2
        rewards.append(min(round(score, 3), 2.0))
    return rewards

def reward_anti_repetition(completions: List[Any], **kwargs) -> List[float]:
    rewards = []
    for item in completions:
        text = extract_text(item)
        if len(text.strip()) < 30:
            rewards.append(-1.5)
            continue
        words = text.lower().split()
        if len(words) >= 16:
            four_grams = [tuple(words[i:i+4]) for i in range(len(words)-3)]
            unique_ratio = len(set(four_grams)) / len(four_grams)
            if unique_ratio < 0.65:
                rewards.append(round((unique_ratio - 0.65) * 2.0, 3))
                continue
        rewards.append(0.0)
    return rewards

print("Reward functions defined successfully!")

### Step 5: Configure GRPOTrainer & LoRA (PEFT)
Includes automated overrides for `torchvision` and `torchao` import compatibility in Colab.

In [ ]:
# 1. Neutralize Colab torchvision & torchao version collisions
import transformers.utils.import_utils as t_utils
t_utils.is_torchvision_available = lambda: False
import peft.import_utils as p_utils
p_utils.is_torchao_available = lambda: False

# 2. CUDA Memory optimizations
import os
import torch
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 3. Library Imports
import polars as pl
from datasets import Dataset
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "grpo_rag_adapter"

# Load training prompts
df = pl.read_parquet("grpo_train_dataset.parquet")
print(f"Loaded {len(df)} total training prompts.")

# Convert to HuggingFace Dataset (use 1000 prompts for Colab T4 session)
dataset = Dataset.from_pandas(df.head(1000).to_pandas())

# LoRA adapter configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

# GRPO configuration (T4 GPU memory-safe optimized)
training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=1.5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,      # Reduced from 8 -> frees ~3.5 GB VRAM
    num_generations=4,                  # Group size G=4
    max_completion_length=256,          # Reduced from 384 -> prevents KV-cache spikes
    max_steps=100,                      # ~35 mins total runtime on T4
    logging_steps=10,
    save_steps=25,                      # Saves checkpoints at 25, 50, 75, 100
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
)

trainer = GRPOTrainer(
    model=MODEL_ID,
    reward_funcs=[reward_reasoning_format, reward_rag_concept_density, reward_anti_repetition],
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
)
print("✅ GRPOTrainer initialized with memory-safe configuration!")

### Step 6: Launch GRPO Reinforcement Learning

In [ ]:
trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f"\nSaved fine-tuned GRPO LoRA adapter to: {OUTPUT_DIR}")

### Step 7: Zip and Download LoRA Adapter

In [ ]:
!zip -r grpo_rag_adapter.zip grpo_rag_adapter
from google.colab import files
files.download("grpo_rag_adapter.zip")
print("Downloaded adapter successfully!")